In [138]:
# Importing Functions from Transformer_Architecture
%run Transformer_Architecture.ipynb

tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])
tensor([[ 9, 14, 16, 20, 16,  0,  0,  0,  0,  0],
        [13,  4, 20, 16, 14, 18, 16, 12,  0,  0],
        [16,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 4, 16,  6, 19, 23,  0,  0,  0,  0,  0]])
tensor([[[-1.1669, -2.2723,  2.3302,  ..., -0.5998, -0.0000, -0.0000],
         [-0.1366, -0.8924,  0.0000,  ..., -2.0140,  0.0000,  1.1805],
         [-0.8538, -3.4220, -2.3415,  ..., -0.0000, -1.8674,  1.1019],
         ...,
         [ 0.6280, -0.8572,  1.3375,  ...,  1.2672,  0.6084, -0.0000],
         [ 2.0105, -0.0279,  0.0208,  ...,  0.0000,  0.1787,  0.0000],
         [ 2.3550,  1.2618, -0.8644,  ...,  3.2807, -0.8245, -0.2389]],

        [[-1.2430, -2.3656,  0.0000,  ...,  0.6028, -0.7281, -1.0450],
         [-1.4553,  0.1875, -2.1840,  ..., -2.2707,  0.1039,  1.4675],
         [ 1.7179, -0.0365, -0.6394,  ..., -2.4836, -1

In [139]:
from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 64
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # Sparse attention
    window_size: int = 4

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [140]:
class PositionalEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.position_embeddings = nn.Embedding(
            num_embeddings=config.max_seq_len,
            embedding_dim=config.d_model,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [B, T]

        returns:
            x: [B, T, d_model]
        """

        batch_size, seq_len = input_ids.shape

        token_embs = self.token_embeddings(input_ids)
        # [B, T, D]

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device,
        )
        # [T]

        pos_embs = self.position_embeddings(position_ids)
        # [T, D]

        x = token_embs + pos_embs
        # [B, T, D]

        return self.dropout(x)

In [141]:
class SlidingWindowCausalSelfAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.d_head = config.d_head
        self.max_seq_len = config.max_seq_len
        self.window_size = config.window_size

        self.q_proj = nn.Linear(config.d_model, config.d_model)
        self.k_proj = nn.Linear(config.d_model, config.d_model)
        self.v_proj = nn.Linear(config.d_model, config.d_model)
        self.o_proj = nn.Linear(config.d_model, config.d_model)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        sparse_mask = self.build_sliding_window_mask(
            max_seq_len=config.max_seq_len,
            window_size=config.window_size,
        )
        # [max_seq_len, max_seq_len]

        sparse_mask = sparse_mask.view(1, 1, config.max_seq_len, config.max_seq_len,)
        # [1, 1, max_seq_len, max_seq_len]

        self.register_buffer("sparse_causal_mask", sparse_mask)

    def build_sliding_window_mask(self, max_seq_len, window_size):
        causal_mask = torch.tril(
            torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
        )
        # allows j <= i
        # Example :  causal_mask = torch.tril(torch.ones(4, 4, dtype=torch.bool)) 
        #tensor([[ True, False, False, False],
        #[ True,  True, False, False],
        #[ True,  True,  True, False],
        #[ True,  True,  True,  True]])
    
        window_mask = torch.triu(
            torch.ones(max_seq_len, max_seq_len, dtype=torch.bool),
            diagonal=-(window_size - 1),
        )
        # allows j >= i - window_size + 1
        # Example : window_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal = -1) 
        # tensor([[ True,  True,  True,  True],
        #[ True,  True,  True,  True],
        #[False,  True,  True,  True],
        #[False, False,  True,  True]])
    
        sparse_mask = causal_mask & window_mask

        # tensor([[ True, False, False, False],
        # [ True,  True, False, False],
        # [False,  True,  True, False],
        # [False, False,  True,  True]])
    
        return sparse_mask

    def forward(self, x, attention_mask=None, return_attn=False):
        """
        x:
            [B, T, d_model]

        attention_mask:
            [B, T], 1 = real token, 0 = padding token

        returns:
            out: [B, T, d_model]
        """

        batch_size, seq_len, d_model = x.shape

        assert d_model == self.d_model
        assert seq_len <= self.max_seq_len

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # [B, T, D]

        q = q.view(batch_size, seq_len, self.n_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_head)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # [B, H, T, d_head]

        scores = q @ k.transpose(-2, -1)
        # [B, H, T, T]

        scores = scores / math.sqrt(self.d_head)

        sparse_mask = self.sparse_causal_mask[:, :, :seq_len, :seq_len]
        # [1, 1, T, T]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            # [B, 1, 1, T]

            combined_mask = sparse_mask & key_padding_mask
            # [B, 1, T, T]
        else:
            combined_mask = sparse_mask
            # [1, 1, T, T]

        scores = scores.masked_fill(
            ~combined_mask,
            torch.finfo(scores.dtype).min,
        )

        attn_weights = torch.softmax(scores, dim=-1)
        # [B, H, T, T]

        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v
        # [B, H, T, d_head]

        out = out.transpose(1, 2).contiguous()
        # [B, T, H, d_head]

        out = out.view(batch_size, seq_len, d_model)
        # [B, T, D]

        out = self.o_proj(out)
        out = self.resid_dropout(out)

        if attention_mask is not None:
            out = out * attention_mask[:, :, None].to(out.dtype)

        if return_attn:
            return out, attn_weights, combined_mask

        return out

In [142]:
class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

In [143]:
class SparseAttentionDecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = SlidingWindowCausalSelfAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, attention_mask=None):
        """
        x: [B, T, d_model]
        """

        x = x + self.attn(
            self.ln_1(x),
            attention_mask=attention_mask,
        )

        x = x + self.mlp(self.ln_2(x))

        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        return x

In [144]:
class SparseAttentionGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = PositionalEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            SparseAttentionDecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:
            [B, T]

        attention_mask:
            [B, T]

        labels:
            [B, T]
        """

        batch_size, seq_len = input_ids.shape

        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask,
            )

        x = self.final_ln(x)

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
        }

    def sample_next_token(
        self,
        next_token_logits,
        temperature=1.0,
        top_k=None,
        top_p=None,
    ):
        if temperature == 0:
            return torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            top_k = min(top_k, next_token_logits.size(-1))

            values, _ = torch.topk(
                next_token_logits,
                k=top_k,
                dim=-1,
            )

            min_values = values[:, -1].unsqueeze(-1)

            next_token_logits = torch.where(
                next_token_logits < min_values,
                torch.full_like(next_token_logits, float("-inf")),
                next_token_logits,
            )

        if top_p is not None:
            assert 0.0 < top_p <= 1.0

            sorted_logits, sorted_indices = torch.sort(
                next_token_logits,
                descending=True,
                dim=-1,
            )

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[:, 1:] = (
                sorted_indices_to_remove[:, :-1].clone()
            )
            sorted_indices_to_remove[:, 0] = False

            indices_to_remove = torch.zeros_like(
                next_token_logits,
                dtype=torch.bool,
            )

            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove,
            )

            next_token_logits = next_token_logits.masked_fill(
                indices_to_remove,
                float("-inf"),
            )

        probs = torch.softmax(next_token_logits, dim=-1)

        return torch.multinomial(
            probs,
            num_samples=1,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens,
        temperature=1.0,
        top_k=None,
        top_p=None,
        eos_token_id=None,
    ):
        """
        Simple autoregressive generation.

        This version recomputes the last max_seq_len tokens each step.
        Later we can add a sparse KV-cache version.
        """

        self.eval()

        for _ in range(max_new_tokens):
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]

            attention_mask = (
                input_ids_cond != self.config.pad_token_id
            ).long()

            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
            )

            next_token_logits = outputs["logits"][:, -1, :]

            next_token = self.sample_next_token(
                next_token_logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

        return input_ids

In [145]:
torch.manual_seed(42)

batch_size = 3
seq_len = 8
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, seq_len),
)

attention_mask = (
    torch.arange(seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(
    attention_mask == 0,
    -100,
)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=16,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,
    window_size=4,
)

model = SparseAttentionGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)

loss.backward()

print("\nGradient exists for q_proj?")
print(model.blocks[0].attn.q_proj.weight.grad is not None)

print("\nGradient exists for k_proj?")
print(model.blocks[0].attn.k_proj.weight.grad is not None)

print("\nGradient exists for v_proj?")
print(model.blocks[0].attn.v_proj.weight.grad is not None)


# -------------------------
# Inspect sparse mask
# -------------------------

print("\nSliding-window causal mask for seq_len=8:")
mask = model.blocks[0].attn.sparse_causal_mask[0, 0, :seq_len, :seq_len]
print(mask.int())


# -------------------------
# Generation test
# -------------------------

prompt_len = int(attention_mask[0].sum().item())
prompt = input_ids[0:1, :prompt_len]

print("\nprompt:")
print(prompt)

generated = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
)

print("\ngenerated:")
print(generated)

print("\ngenerated shape:")
print(generated.shape)

lengths:
tensor([7, 4, 5])

input_ids:
tensor([[59, 91, 66, 26, 78, 86,  3,  0],
        [77, 81, 82, 91,  0,  0,  0,  0],
        [32, 77,  9, 65, 51,  0,  0,  0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0]])

labels:
tensor([[  59,   91,   66,   26,   78,   86,    3, -100],
        [  77,   81,   82,   91, -100, -100, -100, -100],
        [  32,   77,    9,   65,   51, -100, -100, -100]])

logits shape:
torch.Size([3, 8, 100])

loss:
tensor(23.8334, grad_fn=<NllLossBackward0>)

Gradient exists for q_proj?
True

Gradient exists for k_proj?
True

Gradient exists for v_proj?
True

Sliding-window causal mask for seq_len=8:
tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [0, 1, 1, 1, 1, 0, 0, 0],
        [0, 0, 1, 1, 1, 1, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 1, 1, 1, 1]], dtype=torch.int32)

pr

In [146]:
# DeepSeek Sparse Attention Implementation

In [147]:
@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 64
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # MLA-style dimensions
    d_nope: int = 8          # non-RoPE content dim per head
    d_rope: int = 4          # RoPE dim per head, must be even
    r_kv: int = 8            # KV latent dimension
    r_q: int = 16            # Q latent dimension

    # DSA / lightning indexer
    dsa_top_k: int = 4
    indexer_heads: int = 4
    indexer_dim: int = 8

    rope_base: float = 10000.0

    @property
    def d_attn(self):
        return self.d_nope + self.d_rope

In [148]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(
        self,
        rope_dim: int,
        max_seq_len: int,
        base: float = 10000.0,
    ):
        super().__init__()

        assert rope_dim % 2 == 0, "rope_dim must be even for RoPE"

        self.rope_dim = rope_dim
        self.max_seq_len = max_seq_len

        inv_freq = torch.exp(
            torch.arange(0, rope_dim, 2, dtype=torch.float32)
            * (-math.log(base) / rope_dim)
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        angles = positions * inv_freq

        cos = torch.cos(angles)
        sin = torch.sin(angles)

        # Broadcast with x: [B, H, T, rope_dim]
        self.register_buffer("cos", cos[None, None, :, :])
        self.register_buffer("sin", sin[None, None, :, :])

    def forward(self, x):
        """
        x: [B, H, T, rope_dim]
        """

        batch_size, n_heads, seq_len, rope_dim = x.shape

        assert rope_dim == self.rope_dim
        assert seq_len <= self.max_seq_len

        cos = self.cos[:, :, :seq_len, :]
        sin = self.sin[:, :, :seq_len, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd = x_even * sin + x_odd * cos

        x_rot = torch.empty_like(x)
        x_rot[..., 0::2] = x_rot_even
        x_rot[..., 1::2] = x_rot_odd

        return x_rot


In [149]:
def apply_rope_bthd(
    rope: RotaryPositionalEmbedding,
    x: torch.Tensor,
):
    """
    Apply RoPE to [B, T, H, D] using our consistent RoPE class.

    Input:
        x: [B, T, H, D]

    RoPE expects:
        [B, H, T, D]
    """

    x = x.transpose(1, 2)
    # [B, H, T, D]

    x = rope(x)

    x = x.transpose(1, 2)
    # [B, T, H, D]

    return x

In [150]:
class TokenEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [B, T]
        """

        return self.dropout(self.token_embeddings(input_ids))

In [151]:
# ============================================================
# DSA-style Sparse MLA Attention
# ============================================================

class DSASparseMLAAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        assert config.d_rope % 2 == 0, "d_rope must be even for RoPE"

        self.config = config

        self.d_model = config.d_model
        self.num_heads = config.n_heads
        self.d_nope = config.d_nope
        self.d_rope = config.d_rope
        self.d_attn = config.d_attn
        self.r_kv = config.r_kv
        self.r_q = config.r_q

        self.dsa_top_k = config.dsa_top_k
        self.indexer_heads = config.indexer_heads
        self.indexer_dim = config.indexer_dim

        # -------------------------
        # MLA KV path
        # -------------------------

        self.W_DKV = nn.Linear(config.d_model, config.r_kv, bias=False)
        self.kv_norm = nn.LayerNorm(config.r_kv)
        self.W_UK = nn.Linear(config.r_kv, config.n_heads * config.d_nope, bias=False,)
        self.W_UV = nn.Linear(config.r_kv, config.n_heads * config.d_nope, bias=False,)

        # -------------------------
        # MLA Q path
        # -------------------------

        self.W_DQ = nn.Linear(config.d_model, config.r_q, bias=False,)
        self.q_norm = nn.LayerNorm(config.r_q)
        self.W_UQ = nn.Linear(config.r_q, config.n_heads * config.d_nope, bias=False,)

        # -------------------------
        # ROPE Path
        # -------------------------
        self.W_QR = nn.Linear(config.r_q, config.n_heads * config.d_rope, bias=False,)

        self.W_KR = nn.Linear(config.d_model, config.d_rope,bias=False,)

        self.rope = RotaryPositionalEmbedding(
            rope_dim=config.d_rope,
            max_seq_len=config.max_seq_len,
            base=config.rope_base,
        )

        self.W_O = nn.Linear(config.n_heads * config.d_nope,  config.d_model, bias=False,)

        # -------------------------
        # Lightning indexer
        # -------------------------
        # This scores which previous tokens should be selected.
        # It is separate from the main MLA Q/K/V attention path.

        self.index_q_proj = nn.Linear(config.d_model, config.indexer_heads * config.indexer_dim, bias=False,)
        self.index_k_proj = nn.Linear(config.d_model, config.indexer_heads * config.indexer_dim, bias=False,)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def build_mla_qkv(self, x):
        """
        x: [B, T, D]

        returns:
            q: [B, T, H, d_attn]
            k: [B, T, H, d_attn]
            v: [B, T, H, d_nope]
        """

        B, T, D = x.shape

        # -------------------------
        # Q path
        # -------------------------

        c_q = self.W_DQ(x)
        c_q = self.q_norm(c_q)
        # [B, T, r_q]

        q_c = self.W_UQ(c_q).view(B, T, self.num_heads, self.d_nope,)
        # [B, T, H, d_nope]

        q_r = self.W_QR(c_q).view(B, T, self.num_heads, self.d_rope,)
        # [B, T, H, d_rope]

        q_r = apply_rope_bthd(self.rope, q_r)

        q = torch.cat([q_c, q_r], dim=-1)
        # [B, T, H, d_attn]

        # -------------------------
        # KV path
        # -------------------------

        c_kv = self.W_DKV(x)
        c_kv_normed = self.kv_norm(c_kv)
        # [B, T, r_kv]

        k_c = self.W_UK(c_kv_normed).view(B, T, self.num_heads, self.d_nope,)
        # [B, T, H, d_nope]

        v = self.W_UV(c_kv_normed).view(B, T, self.num_heads, self.d_nope,)
        # [B, T, H, d_nope]

        k_r = self.W_KR(x).view(B, T, 1, self.d_rope,)
        # [B, T, 1, d_rope]

        k_r = apply_rope_bthd(self.rope, k_r)

        k_r = k_r.expand(-1, -1, self.num_heads, -1,)
        # [B, T, H, d_rope]

        k = torch.cat([k_c, k_r], dim=-1)
        # [B, T, H, d_attn]

        return q, k, v

    def build_causal_padding_mask(self, seq_len, attention_mask, device):
        """
        returns:
            allowed_mask: [B or 1, T, T], True = allowed
        """

        causal_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device,))
        # [T, T]

        causal_mask = causal_mask[None, :, :]
        # [1, T, T]

        if attention_mask is None:
            return causal_mask

        key_padding_mask = attention_mask[:, None, :].bool()
        # [B, 1, T]

        allowed_mask = causal_mask & key_padding_mask
        # [B, T, T]

        return allowed_mask

    def lightning_indexer_select_mask(self, x, allowed_mask):
        """
        x:
            [B, T, D]

        allowed_mask:
            [B or 1, T, T]

        returns:
            selected_mask: [B, T, T], True = selected key for query
            index_scores:  [B, T, T]
        """

        B, T, D = x.shape
       
        q_idx = self.index_q_proj(x).view(B, T, self.indexer_heads, self.indexer_dim,)
        # [B, T, I, Didx]

        k_idx = self.index_k_proj(x).view(B, T, self.indexer_heads, self.indexer_dim,)
        # [B, T, I, Didx]

        # Indexer score between query t and key s.
        # [B, I, T, S]
        raw_scores = torch.einsum(
            "btid,bsid->bits",
            q_idx,
            k_idx,
        ) / math.sqrt(self.indexer_dim)

        # DeepSeek-style descriptions use ReLU-like positive index scores.
        index_scores_per_head = F.relu(raw_scores)
        # [B, I, T, S]

        # Collapse indexer heads into one selection score.
        index_scores = index_scores_per_head.mean(dim=1)
        # [B, T, S]

        if allowed_mask.shape[0] == 1:
            allowed_mask = allowed_mask.expand(B, -1, -1)

        masked_index_scores = index_scores.masked_fill(
            ~allowed_mask,
            torch.finfo(index_scores.dtype).min,
        )
        # [B, T, S]

        top_k = min(self.dsa_top_k, T)

        _, selected_indices = torch.topk(
            masked_index_scores,
            k=top_k,
            dim=-1,
        )
        # [B, T, top_k]

        selected_mask = torch.zeros(B, T, T, dtype=torch.bool, device=x.device,)

        selected_mask.scatter_(dim=-1, index=selected_indices, value=True,)

        # Remove any invalid positions that got selected because top_k > valid count.
        selected_mask = selected_mask & allowed_mask

        return selected_mask, index_scores

    def forward(
        self,
        x,
        attention_mask=None,
        return_debug=False,
    ):
        """
        x:
            [B, T, d_model]

        attention_mask:
            [B, T], 1 = valid token, 0 = padding

        DSA-style process:
            1. Build dense MLA Q/K/V.
            2. Lightning indexer scores previous tokens.
            3. Select top-k previous tokens per query.
            4. Main attention attends only to selected tokens.
        """

        B, T, D = x.shape
        assert D == self.d_model
        assert T <= self.config.max_seq_len

        q, k, v = self.build_mla_qkv(x)
        # q: [B, T, H, d_attn]
        # k: [B, T, H, d_attn]
        # v: [B, T, H, d_nope]

        allowed_mask = self.build_causal_padding_mask(
            seq_len=T,
            attention_mask=attention_mask,
            device=x.device,
        )
        # [B or 1, T, T]

        selected_mask, index_scores = self.lightning_indexer_select_mask(
            x=x,
            allowed_mask=allowed_mask,
        )
        # selected_mask: [B, T, T]

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        # q: [B, H, T, d_attn]
        # k: [B, H, T, d_attn]
        # v: [B, H, T, d_nope]

        scores = q @ k.transpose(-2, -1)
        # [B, H, T, T]

        scores = scores / math.sqrt(self.d_attn)

        scores = scores.masked_fill(
            ~selected_mask[:, None, :, :],
            torch.finfo(scores.dtype).min,
        )

        attn = torch.softmax(scores, dim=-1)
        # [B, H, T, T]

        attn = self.attn_dropout(attn)

        out = attn @ v
        # [B, H, T, d_nope]

        out = out.transpose(1, 2).contiguous()
        # [B, T, H, d_nope]

        out = out.view(
            B,
            T,
            self.num_heads * self.d_nope,
        )
        # [B, T, H * d_nope]

        out = self.W_O(out)
        out = self.resid_dropout(out)

        if attention_mask is not None:
            out = out * attention_mask[:, :, None].to(out.dtype)

        if return_debug:
            return {
                "out": out,
                "selected_mask": selected_mask,
                "index_scores": index_scores,
                "attn": attn,
            }

        return out

In [152]:
# FeedForward + Decoder Block
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

In [153]:
class DSADecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = DSASparseMLAAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, attention_mask=None, return_debug=False):
        if return_debug:
            debug = self.attn(
                self.ln_1(x),
                attention_mask=attention_mask,
                return_debug=True,
            )
            attn_out = debug["out"]
        else:
            attn_out = self.attn(
                self.ln_1(x),
                attention_mask=attention_mask,
                return_debug=False,
            )
            debug = None

        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))

        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        if return_debug:
            return x, debug

        return x

In [154]:
# ============================================================
# DSA GPT Decoder
# ============================================================

class DSAGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            DSADecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Weight tying
        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        return_debug=False,
    ):
        """
        input_ids:
            [B, T]

        attention_mask:
            [B, T]

        labels:
            [B, T]
        """

        B, T = input_ids.shape
        assert T <= self.config.max_seq_len

        x = self.embedding(input_ids)

        debug_by_layer = []

        for block in self.blocks:
            if return_debug:
                x, debug = block(
                    x,
                    attention_mask=attention_mask,
                    return_debug=True,
                )
                debug_by_layer.append(debug)
            else:
                x = block(
                    x,
                    attention_mask=attention_mask,
                    return_debug=False,
                )

        x = self.final_ln(x)

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )

        return {
            "logits": logits,
            "loss": loss,
            "debug_by_layer": debug_by_layer,
        }

    def sample_next_token(
        self,
        next_token_logits,
        temperature=1.0,
        top_k=None,
        top_p=None,
    ):
        if temperature == 0:
            return torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            top_k = min(top_k, next_token_logits.size(-1))

            values, _ = torch.topk(
                next_token_logits,
                k=top_k,
                dim=-1,
            )

            min_values = values[:, -1].unsqueeze(-1)

            next_token_logits = torch.where(
                next_token_logits < min_values,
                torch.full_like(next_token_logits, float("-inf")),
                next_token_logits,
            )

        if top_p is not None:
            assert 0.0 < top_p <= 1.0

            sorted_logits, sorted_indices = torch.sort(
                next_token_logits,
                descending=True,
                dim=-1,
            )

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[:, 1:] = (
                sorted_indices_to_remove[:, :-1].clone()
            )
            sorted_indices_to_remove[:, 0] = False

            indices_to_remove = torch.zeros_like(
                next_token_logits,
                dtype=torch.bool,
            )

            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove,
            )

            next_token_logits = next_token_logits.masked_fill(
                indices_to_remove,
                float("-inf"),
            )

        probs = torch.softmax(next_token_logits, dim=-1)

        return torch.multinomial(
            probs,
            num_samples=1,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens,
        temperature=1.0,
        top_k=None,
        top_p=None,
        eos_token_id=None,
    ):
        """
        Simple generation.

        This recomputes the full prefix each step.
        It still uses DSA-style sparse token selection inside attention.
        """

        self.eval()

        for _ in range(max_new_tokens):
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]

            attention_mask = (
                input_ids_cond != self.config.pad_token_id
            ).long()

            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
                return_debug=False,
            )

            next_token_logits = outputs["logits"][:, -1, :]

            next_token = self.sample_next_token(
                next_token_logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

        return input_ids

In [155]:
torch.manual_seed(42)

batch_size = 3
seq_len = 8
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, seq_len),
)

attention_mask = (
    torch.arange(seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(
    attention_mask == 0,
    -100,
)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=32,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,

    d_nope=8,
    d_rope=4,
    r_kv=8,
    r_q=16,

    dsa_top_k=4,
    indexer_heads=4,
    indexer_dim=8,
)

model = DSAGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    return_debug=True,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)

loss.backward()

print("\nGradient exists for W_DQ?")
print(model.blocks[0].attn.W_DQ.weight.grad is not None)

print("\nGradient exists for W_DKV?")
print(model.blocks[0].attn.W_DKV.weight.grad is not None)

print("\nGradient exists for lightning indexer query projection?")
print(model.blocks[0].attn.index_q_proj.weight.grad is not None)

print("\nGradient exists for lightning indexer key projection?")
print(model.blocks[0].attn.index_k_proj.weight.grad is not None)


# -------------------------
# Inspect selected sparse mask
# -------------------------

debug0 = outputs["debug_by_layer"][0]
selected_mask = debug0["selected_mask"]
# [B, T, T]

print("\nSelected mask for example 0:")
print(selected_mask[0].int())

print("\nNumber of selected keys per query for example 0:")
print(selected_mask[0].sum(dim=-1))


# -------------------------
# Generation test
# -------------------------

prompt_len = int(attention_mask[0].sum().item())
prompt = input_ids[0:1, :prompt_len]

print("\nprompt:")
print(prompt)

generated = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
)

print("\ngenerated:")
print(generated)

print("\ngenerated shape:")
print(generated.shape)

lengths:
tensor([7, 4, 5])

input_ids:
tensor([[59, 91, 66, 26, 78, 86,  3,  0],
        [77, 81, 82, 91,  0,  0,  0,  0],
        [32, 77,  9, 65, 51,  0,  0,  0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0]])

labels:
tensor([[  59,   91,   66,   26,   78,   86,    3, -100],
        [  77,   81,   82,   91, -100, -100, -100, -100],
        [  32,   77,    9,   65,   51, -100, -100, -100]])

logits shape:
torch.Size([3, 8, 100])

loss:
tensor(32.5934, grad_fn=<NllLossBackward0>)

Gradient exists for W_DQ?
True

Gradient exists for W_DKV?
True

Gradient exists for lightning indexer query projection?
False

Gradient exists for lightning indexer key projection?
False

Selected mask for example 0:
tensor([[1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 1, 0, 0],
        [1, 0, 0,

In [156]:
# Test code to understand Lightening Indexer:

import torch
import torch.nn.functional as F
import math

# 1. Initialize our shapes and parameters
B, T, D = 1, 3, 4
indexer_heads = 2
indexer_dim = 2
dsa_top_k = 2

# 2. Input Tensor (X) - Shape: [1, 3, 4]
X = torch.tensor([[[1.0, 0.0, 2.0, 0.0],
                   [0.0, 2.0, 0.0, 1.0],
                   [1.0, 1.0, 1.0, 1.0]]])

# 3. Weights Matrix Transposed (W_T) - Shape: [4, 4]
# (Simulating an nn.Linear weight parameter matrix)
W_T = torch.tensor([[1.0, 0.0, 1.0, 0.0],
                    [0.0, 1.0, 0.0, 1.0],
                    [1.0, 1.0, 0.0, 0.0],
                    [0.0, 0.0, 1.0, 1.0]])

# 4. Step 1: Linear Projection via torch.matmul
# Shape: [1, 3, 4] x [4, 4] -> [1, 3, 4]
proj_out = torch.matmul(X, W_T)

# 5. Step 2 & 3: Reshaping and Permuting into Head dimensions
# Shape change: [1, 3, 4] -> .view() -> [1, 3, 2, 2] -> .permute() -> [1, 2, 3, 2]
q_idx = proj_out.view(B, T, indexer_heads, indexer_dim).permute(0, 2, 1, 3)
k_idx = q_idx.clone() # Using identical values for keys to match previous math

# 6. Step 4: Compute Dot Product Matrix via torch.matmul
# We transpose the last two dimensions of K to align matrix multiplication
# Shape: [1, 2, 3, 2] x [1, 2, 2, 3] -> [1, 2, 3, 3]
k_idx_T = k_idx.transpose(-2, -1)
raw_scores = torch.matmul(q_idx, k_idx_T) / math.sqrt(indexer_dim)

# 7. Step 5: Activation and Pooling (Mean reduction across Head dimension)
# Shape change: [1, 2, 3, 3] -> F.relu() -> [1, 2, 3, 3] -> .mean(dim=1) -> [1, 3, 3]
activated_scores = F.relu(raw_scores)
index_scores = activated_scores.mean(dim=1)

# 8. Step 6: Causal Masking
# Create a standard lower-triangular causal matrix
allowed_mask = torch.tril(torch.ones(T, T, dtype=torch.bool)).unsqueeze(0) # [1, 3, 3]

# Fill future tokens with -inf
masked_index_scores = index_scores.masked_fill(~allowed_mask, float('-inf'))

# 9. Step 7: Top-k Selection
# Shape: [1, 3, 3] -> torch.topk() -> [1, 3, 2]
top_k = min(dsa_top_k, T)
_, selected_indices = torch.topk(masked_index_scores, k=top_k, dim=-1)

# 10. Step 8: Scatter Mask Reconstruction and Cleanup
# Build an empty all-False tensor, scatter True values, and perform a bitwise AND
selected_mask = torch.zeros(B, T, T, dtype=torch.bool)
selected_mask.scatter_(dim=-1, index=selected_indices, value=True)
final_mask = selected_mask & allowed_mask

# Print the final result matrix
print(final_mask)

tensor([[[ True, False, False],
         [ True,  True, False],
         [ True, False,  True]]])
